# Quinta Playa — multi-month beach change pipeline (py4dgeo)

Full pipeline: stable-ground registration check across all months → pairwise M3C2 →
4D objects-by-change time series.

**Run order matters.** Phase 1 is a *diagnostic* — it tells you whether the months are
on a common frame at all. Don't skip to Phase 3 before reading Phase 1's output.

### Findings carried in from the Jan→Feb 2025 investigation

- **Horizontal registration is excellent, vertical is not.** Across four stable features:
  horizontal agreement 2–15 mm, vertical 4–33 cm. This is expected (nadir photogrammetry
  resolves Z from parallax at shallow angles) but the *spread* is the problem.
- **No coherent site-wide vertical offset was found.** The three rocks gave negative dz,
  the rooftop positive. A real datum shift would push every stable feature the same way.
- **Neither distance-from-GCP (doming) nor point count explained the spread.** Both were
  tested and both failed. The cause is still open.
- **Practical consequence:** treat vertical change below ~±20 cm as not reliably
  distinguishable for this data. Horizontal/planform change is trustworthy to centimetres.

### Prerequisites

- **Dec 2024 must have its `set1_to_set2` GCP correction applied** before being used here
  (it won its A/B test with `ground_control_points.txt`; everything else is on the
  `GCP_2025.txt` frame). Without this it will show a spurious ~1.8 m offset.
- Point clouds exported from Metashape as **dense cloud**, not mesh.
- Water/surf-zone noise filtered before export, or expect spurious spikes.


## Phase 0 — Configuration

Everything you need to edit lives in this one cell.


In [ ]:
import gc
import itertools
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import py4dgeo
from scipy.spatial import cKDTree

print("py4dgeo", py4dgeo.__version__ if hasattr(py4dgeo, "__version__") else "loaded")

# --- Months, in chronological order. Comment out ones not yet processed. ---
MONTHS = {
    "Dec2024": "E:/Finalised/2025/December/PointCloud_Dec_2025.laz",   # NEEDS GCP set1->set2 correction
    "Jan2025": "E:/Finalised/2025/January/PointCloud_Jan_2025.laz",
    "Feb2025": "E:/Finalised/2025/February/PointCloud_Feb_2025.laz",
    # "Mar2025": ...,
    # "Apr2025": ...,
    # "May2025": ...,
}

# --- Reference month: every other month is registered against this one.
# Pick the one with the best GCP fit and the most complete coverage. ---
REFERENCE = "Jan2025"

# --- Stable features for the registration check.
# Rock 1 was the most trustworthy in the Jan->Feb tests (dense, tight residual,
# near-zero rotation). Keep several: if one isn't in a month's flight footprint,
# you still have the others, and disagreement between them is itself diagnostic. ---
STABLE_FEATURES = {
    "rock1":   ((714334.86, 9888847.23), (714344.00, 9888851.06)),
    "rock2":   ((712936.0,  9888565.0),  (712946.0,  9888575.0)),   # EDIT: real corners
    "rock3":   ((713031.0,  9888581.0),  (713041.0,  9888591.0)),   # EDIT: real corners
    "rooftop": ((713014.0,  9888682.0),  (713025.0,  9888692.0)),   # EDIT: real corners
}
PRIMARY_FEATURE = "rock1"   # the one used to actually correct, if you correct at all

# --- M3C2 parameters. Both still need real tuning (see Phase 3). ---
NORMAL_RADII = (0.5,)
CYL_RADIUS   = 1.0
COREPOINT_SPACING = 1.0     # metres; voxel size for subsampling corepoints

# --- Optional: restrict analysis to the beach itself.
# Set to None to use the full footprint, or ((xmin,ymin),(xmax,ymax)). ---
BEACH_BBOX = None

OUTDIR = Path("results")
OUTDIR.mkdir(exist_ok=True)


## Phase 0b — Helpers

`apply_transform` deliberately uses py4dgeo's own `Epoch.transform()` rather than
multiplying the matrix by hand. The matrix must be applied as `R(x − x₀) + t + x₀`
where `x₀` is the reduction point — applying `Rx + t` and ignoring `x₀` gives a
wrong answer, and how wrong depends on the rotation, so it can look plausible
while being off by tens of centimetres in Z. Let the library do it.


In [ ]:
def crop(cloud, bbox):
    """Crop an (N,3) array to a ((xmin,ymin),(xmax,ymax)) box."""
    (xmin, ymin), (xmax, ymax) = bbox
    m = (
        (cloud[:, 0] >= xmin) & (cloud[:, 0] <= xmax)
        & (cloud[:, 1] >= ymin) & (cloud[:, 1] <= ymax)
    )
    return cloud[m]


def apply_transform(cloud, trafo):
    """Apply a py4dgeo Transformation to a raw array, reduction point included."""
    ep = py4dgeo.Epoch(cloud.copy())
    ep.transform(transformation=trafo)
    return ep.cloud


def register(ref_cloud, moving_cloud):
    """ICP moving -> ref. Returns (trafo, dx, dy, dz, residual_mean, residual_std).

    py4dgeo convention: iterative_closest_point(reference, moving) returns the
    transformation that maps `moving` onto `reference`. So a dz of -0.04 means
    `moving` must drop 4 cm to match `reference` -- i.e. moving sits 4 cm high.
    """
    ref = py4dgeo.Epoch(ref_cloud)
    mov = py4dgeo.Epoch(moving_cloud)
    red = ref_cloud.mean(axis=0)
    trafo = py4dgeo.iterative_closest_point(
        ref, mov, tolerance=0.00001, max_iterations=50, reduction_point=red
    )
    moved = apply_transform(moving_cloud, trafo)
    d, _ = cKDTree(ref_cloud).query(moved)
    dx, dy, dz = trafo.affine_transformation[:3, 3]
    return trafo, dx, dy, dz, d.mean(), d.std()


def point_spacing(cloud, sample=20000):
    """Mean nearest-neighbour distance -- the cloud's own native resolution.

    Residuals are only comparable between features after dividing by this: a
    sparser cloud has larger nearest-neighbour distances even when perfectly
    registered, so raw residuals penalise sparse crops unfairly.
    """
    if cloud.shape[0] > sample:
        cloud = cloud[np.random.default_rng(0).choice(cloud.shape[0], sample, replace=False)]
    d, _ = cKDTree(cloud).query(cloud, k=2)
    return d[:, 1].mean()


## Phase 1 — Stable-ground registration across all months

Loads each month one at a time, keeps only the stable-feature crops, and frees the
full cloud immediately. The crops are tiny (tens of thousands of points), so every
month's crops fit in memory at once even though two full clouds nearly killed a
16 GB machine.

This cell is the memory-heavy one. If the kernel dies here, run this phase as a
standalone `.py` script instead — the VS Code Jupyter kernel has been less stable
than a plain `python` process on this data.


In [ ]:
crops = {feat: {} for feat in STABLE_FEATURES}

for name, path in MONTHS.items():
    print(f"\nLoading {name} ...")
    ep = py4dgeo.read_from_las(path)
    print(f"  full cloud: {ep.cloud.shape[0]:,} points")
    for feat, bbox in STABLE_FEATURES.items():
        c = crop(ep.cloud, bbox)
        if c.shape[0] == 0:
            print(f"  {feat:<10} NOT VISIBLE in this month's footprint -- skipping")
            continue
        crops[feat][name] = c
        print(f"  {feat:<10} {c.shape[0]:>8,} points   spacing {point_spacing(c):.3f} m")
    del ep
    gc.collect()


### 1a — All pairs, per feature

Not sequential chaining. Every month is compared against every other month directly,
so errors don't compound the way Dec→Jan→Feb→Mar would.


In [ ]:
results = {}   # (feature, month_a, month_b) -> dict

for feat, by_month in crops.items():
    months_here = [m for m in MONTHS if m in by_month]
    for a, b in itertools.combinations(months_here, 2):
        trafo, dx, dy, dz, res, res_sd = register(by_month[a], by_month[b])
        spacing = point_spacing(by_month[a])
        results[(feat, a, b)] = dict(
            trafo=trafo, dx=dx, dy=dy, dz=dz,
            res=res, res_sd=res_sd,
            rel_res=res / spacing,
            n_ref=by_month[a].shape[0], n_mov=by_month[b].shape[0],
        )

hdr = f"{'feature':<10}{'pair':<20}{'horiz (cm)':>11}{'dz (cm)':>10}{'resid':>9}{'resid/spacing':>15}"
print(hdr)
print("-" * len(hdr))
for (feat, a, b), r in results.items():
    horiz = np.hypot(r["dx"], r["dy"]) * 100
    print(f"{feat:<10}{a+'->'+b:<20}{horiz:>11.1f}{r['dz']*100:>10.1f}"
          f"{r['res']:>9.3f}{r['rel_res']:>15.2f}")


### 1b — Do the features agree with each other?

For a given month pair, every stable feature should report the same offset. They
didn't for Jan→Feb (4 to 33 cm, and not even the same sign). Spread across features
is your real uncertainty — bigger than any single feature's residual suggests.


In [ ]:
print(f"{'pair':<20}{'feature':<10}{'dz (cm)':>10}   <- spread across features is the real uncertainty")
print("-" * 62)
for a, b in itertools.combinations(list(MONTHS), 2):
    dzs = []
    for feat in STABLE_FEATURES:
        r = results.get((feat, a, b))
        if r is None:
            continue
        dzs.append(r["dz"] * 100)
        print(f"{a+'->'+b:<20}{feat:<10}{r['dz']*100:>10.1f}")
    if len(dzs) > 1:
        print(f"{'':<20}{'SPREAD':<10}{max(dzs)-min(dzs):>10.1f}  "
              f"(min {min(dzs):.1f}, max {max(dzs):.1f})")
    print()


### 1c — Triangle closure

If the measurements are self-consistent, `A→B` plus `B→C` should equal `A→C`, so
closure lands near zero. This is the check that makes all-pairs worth doing over
sequential chaining — it validates the numbers against themselves.

Closure of a few cm: offsets are real and consistent. Closure of 20+ cm: noise
dominates and the individual numbers shouldn't be trusted as corrections.


In [ ]:
for feat in STABLE_FEATURES:
    months_here = [m for m in MONTHS if m in crops.get(feat, {})]
    if len(months_here) < 3:
        continue
    print(f"\n{feat}:")
    for a, b, c in itertools.combinations(months_here, 3):
        try:
            ab = results[(feat, a, b)]["dz"]
            bc = results[(feat, b, c)]["dz"]
            ac = results[(feat, a, c)]["dz"]
        except KeyError:
            continue
        print(f"  {a}->{b}->{c}: closure {(ab + bc - ac) * 100:+.1f} cm")


### 1d — Decide

Read the three tables above before continuing.

- **Spread across features < ~5 cm and closure < ~5 cm** → the months are on a
  common frame. Apply the `PRIMARY_FEATURE` correction (or nothing) and carry on.
- **Spread is large (the Jan→Feb situation)** → there is no single correct vertical
  correction. Applying one anyway would be picking a number for its convenience.
  Carry on to M3C2, but treat vertical results below the spread as noise, and say
  so in anything you report.

`APPLY_VERTICAL_CORRECTION = False` is the honest default given what Jan→Feb showed.
Set it True only if the diagnostics above actually support it.


In [ ]:
APPLY_VERTICAL_CORRECTION = False

corrections = {}   # month -> Transformation to apply on load
if APPLY_VERTICAL_CORRECTION:
    for m in MONTHS:
        if m == REFERENCE:
            continue
        key = (PRIMARY_FEATURE, REFERENCE, m) if (PRIMARY_FEATURE, REFERENCE, m) in results \
              else (PRIMARY_FEATURE, m, REFERENCE)
        if key in results:
            corrections[m] = results[key]["trafo"]
            print(f"{m}: will apply correction from {PRIMARY_FEATURE} "
                  f"({results[key]['dz']*100:+.1f} cm dz)")
else:
    print("No correction will be applied -- months used as-is.")
    print("Vertical results should be interpreted against the feature spread above.")


## Phase 2 — Common corepoints

Every month must be measured at **the same** locations, or the time series isn't
comparable. Build them once from the reference month and reuse throughout.

`cloud[::50]` is not a substitute: it keeps every 50th point in *file order*, which
follows scan lines or tiles rather than ground position, so coverage ends up uneven.
It also left ~3.7 M corepoints on a 184 M-point cloud — enough to run for hours.
Voxel subsampling gives genuinely even spacing and a tractable count.


In [ ]:
py4dgeo.enable_trace(False)
py4dgeo.enable_timeit(False)

ref_epoch = py4dgeo.read_from_las(MONTHS[REFERENCE])
print(f"{REFERENCE}: {ref_epoch.cloud.shape[0]:,} points")

vapc = py4dgeo.Vapc(ref_epoch, voxel_size=COREPOINT_SPACING)
corepoints = vapc.reduce_to_feature("closest_to_centroids").epoch.cloud

if BEACH_BBOX is not None:
    corepoints = crop(corepoints, BEACH_BBOX)

print(f"corepoints: {corepoints.shape[0]:,} at {COREPOINT_SPACING} m spacing")
np.save(OUTDIR / "corepoints.npy", corepoints)

del vapc
gc.collect()


## Phase 3 — Pairwise M3C2 between consecutive months

M3C2 gives a **signed** distance along the local surface normal, so positive and
negative mean accretion and erosion — unlike C2C, which only gives magnitude.

Confirm the sign convention on a spot you know changed before trusting the whole map.

### On the parameters

`NORMAL_RADII` and `CYL_RADIUS` still need real tuning. The proper method (Lague et
al. 2013) is a roughness-vs-scale plot: compute local roughness at several radii and
pick the radius where it stops climbing steeply. 0.5 m / 1.0 m are defensible starting
values for beach sand — they match the DEM/ortho resolution used elsewhere in this
project — but they are starting values, not answers.


In [ ]:
month_list = list(MONTHS)
pairs = list(zip(month_list[:-1], month_list[1:]))   # consecutive only

m3c2_results = {}

for a, b in pairs:
    print(f"\n=== {a} -> {b} ===")
    ep_a = py4dgeo.read_from_las(MONTHS[a])
    ep_b = py4dgeo.read_from_las(MONTHS[b])

    for m, ep in ((a, ep_a), (b, ep_b)):
        if m in corrections:
            ep.transform(transformation=corrections[m])
            print(f"  applied correction to {m}")

    m3c2 = py4dgeo.M3C2(
        epochs=(ep_a, ep_b),
        corepoints=corepoints,
        normal_radii=NORMAL_RADII,
        cyl_radius=CYL_RADIUS,
    )
    distances, uncertainties = m3c2.run()
    m3c2_results[(a, b)] = distances
    np.save(OUTDIR / f"m3c2_{a}_{b}.npy", distances)

    finite = np.isfinite(distances)
    print(f"  {finite.sum():,} / {len(distances):,} corepoints returned a distance")
    print(f"  median {np.nanmedian(distances[finite]):+.3f} m   "
          f"p5 {np.nanpercentile(distances[finite], 5):+.3f}   "
          f"p95 {np.nanpercentile(distances[finite], 95):+.3f}")

    del ep_a, ep_b, m3c2
    gc.collect()


### 3a — Maps

Colour scale is clipped to the 5th/95th percentile so a handful of outliers (surf
zone spikes, vegetation) don't flatten the real signal into one colour.


In [ ]:
n = len(m3c2_results)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5.5), squeeze=False)

for ax, ((a, b), distances) in zip(axes[0], m3c2_results.items()):
    finite = np.isfinite(distances)
    lim = np.nanpercentile(np.abs(distances[finite]), 95)
    sc = ax.scatter(
        corepoints[finite, 0], corepoints[finite, 1],
        c=distances[finite], cmap="RdBu_r", vmin=-lim, vmax=lim, s=2,
    )
    ax.set_aspect("equal")
    ax.set_title(f"{a} -> {b}")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain", useOffset=False)
    plt.colorbar(sc, ax=ax, label="M3C2 distance (m)")

plt.tight_layout()
plt.savefig(OUTDIR / "m3c2_maps.png", dpi=150)
plt.show()


### 3b — Distributions

Over a whole beach including unchanged ground, the distribution should centre near
zero. A centre offset well away from zero is a registration problem showing up as
apparent change everywhere — compare it against the Phase 1 feature spread before
reading it as real accretion or erosion.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for (a, b), distances in m3c2_results.items():
    finite = np.isfinite(distances)
    ax.hist(distances[finite], bins=200, range=(-2, 2),
            histtype="step", label=f"{a}->{b}", linewidth=1.4)
ax.axvline(0, color="k", linewidth=0.8, linestyle="--")
ax.set_xlabel("M3C2 distance (m)")
ax.set_ylabel("corepoint count")
ax.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "m3c2_histograms.png", dpi=150)
plt.show()

for (a, b), distances in m3c2_results.items():
    finite = np.isfinite(distances)
    print(f"{a}->{b}: median {np.nanmedian(distances[finite]):+.3f} m  "
          f"(a non-zero centre here is a registration signal, not beach change)")


## Phase 4 — 4D objects-by-change (time series)

### Read this before running

4D-OBC was designed for **high-frequency** time series — the original papers use
hourly terrestrial laser scans, hundreds to thousands of epochs. It grows regions in
space *and time*, and with 6 monthly epochs there is very little time dimension for
it to work with.

So treat this phase as **exploratory**, not as the headline result. The pairwise M3C2
maps in Phase 3 are the defensible output for a dataset this size. The tutorial's
`window_width=6, minperiod=3` are meaningless here and are reduced below; even so,
expect few or no objects, and don't read significance into what does come out.

If the whole season is eventually processed and you want real time-series analysis,
the honest options are seasonal aggregation, or simply stacking the pairwise M3C2
results and looking at cumulative change per corepoint (the cell after next).


In [ ]:
from datetime import datetime

# EDIT: real survey dates. Approximate ones are fine -- only the ordering and
# rough spacing matter -- but they must be correct in order.
TIMESTAMPS = {
    "Dec2024": datetime(2024, 12, 19),
    "Jan2025": datetime(2025, 1, 15),
    "Feb2025": datetime(2025, 2, 15),
    # "Mar2025": datetime(2025, 3, 15),
}

archive = OUTDIR / "quintaplaya_4dobc.zip"
analysis = py4dgeo.SpatiotemporalAnalysis(str(archive), force=True)  # force=True: corepoints can only be set once

ref = py4dgeo.read_from_las(MONTHS[REFERENCE])
if REFERENCE in corrections:
    ref.transform(transformation=corrections[REFERENCE])
ref.timestamp = TIMESTAMPS[REFERENCE]

analysis.reference_epoch = ref
analysis.corepoints = corepoints
analysis.m3c2 = py4dgeo.M3C2(cyl_radius=CYL_RADIUS, normal_radii=list(NORMAL_RADII))

del ref
gc.collect()

for name, path in MONTHS.items():
    if name == REFERENCE:
        continue
    print(f"adding {name} ...")
    ep = py4dgeo.read_from_las(path)
    if name in corrections:
        ep.transform(transformation=corrections[name])
    ep.timestamp = TIMESTAMPS[name]
    analysis.add_epochs(ep)
    del ep
    gc.collect()

print("\nspace-time array:", analysis.distances.shape, "(corepoints x epochs)")
print("timedeltas:", analysis.timedeltas)


In [ ]:
algo = py4dgeo.RegionGrowingAlgorithm(
    neighborhood_radius=2.0,
    seed_subsampling=30,
    window_width=3,      # tutorial uses 6 -- far too long for monthly epochs
    minperiod=2,         # tutorial uses 3 -- ditto
    height_threshold=0.20,   # set at the Phase 1 feature spread, not lower
)

objects = algo.run(analysis)
print(f"extracted {len(objects)} 4D objects-by-change")

if objects:
    objects[0].plot()
    seed_xy = analysis.corepoints.cloud[objects[0].seed.index]
    print(f"largest object seed at {seed_xy[0]:.1f}, {seed_xy[1]:.1f}")
    print(f"  epochs {objects[0].seed.start_epoch} -> {objects[0].seed.end_epoch}")
else:
    print("No objects found -- expected with this few epochs. See the note above.")


### 4a — Cumulative change (the practical alternative)

With few epochs this is more informative than 4D-OBC: total change per corepoint
across the whole period, straight from the space-time array.


In [ ]:
space_time = analysis.distances     # (corepoints, epochs), vs the reference epoch
cumulative = space_time[:, -1]      # last epoch vs reference

finite = np.isfinite(cumulative)
lim = np.nanpercentile(np.abs(cumulative[finite]), 95)

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(corepoints[finite, 0], corepoints[finite, 1],
                c=cumulative[finite], cmap="RdBu_r", vmin=-lim, vmax=lim, s=2)
ax.set_aspect("equal")
ax.set_title(f"Cumulative change: {REFERENCE} -> {list(MONTHS)[-1]}")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.ticklabel_format(style="plain", useOffset=False)
plt.colorbar(sc, ax=ax, label="M3C2 distance (m)")
plt.tight_layout()
plt.savefig(OUTDIR / "cumulative_change.png", dpi=150)
plt.show()

np.save(OUTDIR / "space_time_array.npy", space_time)


## Phase 5 — What to trust

Before this goes anywhere near a report or the Charles Darwin Foundation:

1. **Quote the feature spread from Phase 1 as the uncertainty**, not the residual of
   whichever single feature fit best. The spread is the honest number.
2. **Horizontal/planform change is the defensible result.** Shoreline position,
   beach width, planform area — these agreed to centimetres across every test.
3. **Vertical change below the spread is not resolvable.** Volumetric estimates
   inherit this directly, so either report them with that uncertainty attached or
   don't report them.
4. **The vertical inconsistency is itself a finding worth reporting.** Combined with
   the per-month GCP file inconsistency already in the plan doc, it supports the
   recommendation to set the D-RTK 2 base station over a fixed benchmark rather than
   re-averaging each session.

### Open

- Cause of the vertical spread. Distance-from-GCP and point count were both tested
  and neither explained it. Next step is visual: inspect each stable feature in the
  orthomosaic for anything that genuinely changed (debris, vegetation, a fixture)
  that a bounding box wouldn't exclude.
- Real M3C2 parameter tuning via a roughness-vs-scale analysis.
- Cross-check against the geodesic profile script (`MatanYuval/WorkWithMaca`) on a
  few transects, as an independent second opinion.
